In [1]:
import polars as pl 
from dbconfig import engine
print('Environment ready!')

Environment ready!


In [2]:
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_width_chars(180)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_float_precision(2)

polars.config.Config

In [4]:
pl.read_database(
        query = """
        select table_name, column_name
        from information_schema.columns 
        where table_schema = 'analysis'
        order by table_name, ordinal_position;
        """, connection = engine
        )

table_name,column_name
str,str
"""inflation""","""year"""
"""inflation""","""inflation_rate"""
"""nifty_monthly""","""date"""
"""nifty_monthly""","""price"""
"""nifty_monthly""","""open"""
"""nifty_monthly""","""high"""
"""nifty_monthly""","""low"""
"""nifty_monthly""","""volume"""
"""nifty_monthly""","""change_pct"""


In [5]:
inflation_df = pl.read_database(
        query = """
        select * 
        from analysis.inflation 
        order by year;
        """, connection = engine
        )

In [6]:
inflation_df.head()

year,inflation_rate
i64,f64
1995,10.22
1996,8.98
1997,7.16
1998,13.23
1999,4.67


In [7]:
nifty_monthly_df = pl.read_database(
        query = """
        select *
        from analysis.nifty_monthly 
        order by date;
        """, connection = engine
        )

In [8]:
nifty_monthly_df.head()

date,price,open,high,low,volume,change_pct
date,f64,f64,f64,f64,f64,f64
1995-12-01,908.53,867.99,927.60,854.47,null,5.39
1996-01-01,848.42,913.11,913.11,813.12,null,-6.62
1996-02-01,992.51,848.28,1067.49,848.28,null,16.98
1996-03-01,985.30,994.66,1010.50,945.07,null,-0.73
1996-04-01,1114.36,988.33,1160.16,988.33,null,13.10


In [36]:
def simulate_historical(
    nifty_50: pl.DataFrame,
    inflation_rate: pl.DataFrame,
    contribution_growth: float,
    initial_monthly: float = 5000.0,
) -> pl.DataFrame:

    inflation_lookup = {
        row["year"]: row["inflation_rate"] / 100
        for row in inflation_rate.iter_rows(named=True)
    }

    monthly_contribution = initial_monthly

    total_units = 0.0
    total_contributions = 0.0
    total_real_contributions = 0.0

    cumulative_inflation_factor = 1.0
    current_year = None

    rows = []

    for row in nifty_50.iter_rows(named=True):

        date = row["date"]
        price = row["price"]
        year = date.year

        if current_year is None:
            current_year = year

        elif year != current_year:

            cumulative_inflation_factor *= (
                1 + inflation_lookup[current_year]
            )

            monthly_contribution *= (
                1 + (
                    inflation_lookup[current_year]
                    * contribution_growth
                )
            )

            current_year = year

        units_bought = monthly_contribution / price
        total_units += units_bought

        total_contributions += monthly_contribution

        nominal_corpus = total_units * price

        investment_gain = (
            nominal_corpus - total_contributions
        )

        real_contribution = (
            monthly_contribution /
            cumulative_inflation_factor
        )

        total_real_contributions += real_contribution

        real_corpus = (
            nominal_corpus /
            cumulative_inflation_factor
        )

        real_wealth_created = (
            real_corpus -
            total_real_contributions
        )

        purchasing_power_retained = (
            real_corpus /
            total_real_contributions
        )

        rows.append({
            "date": date,
            "price": round(price, 2),
            "monthly_contribution": round(monthly_contribution, 2),
            "units_bought": round(units_bought, 6),
            "total_units": round(total_units, 6),
            "total_contributions": round(total_contributions, 2),
            "real_contributions": round(total_real_contributions, 2),
            "investment_gain": round(investment_gain, 2),
            "inflation_factor": round(cumulative_inflation_factor, 4),
            "nominal_corpus": round(nominal_corpus, 2),
            "real_corpus": round(real_corpus, 2),
            "real_wealth_created": round(real_wealth_created, 2),
            "purchasing_power_retained": round(
                purchasing_power_retained * 100,
                2,
            ),
        })

    return pl.DataFrame(rows)

In [40]:
nifty_historical_fixed = simulate_historical(
nifty_monthly_df, inflation_df, contribution_growth = 0.0
        )

In [41]:
nifty_historical_half = simulate_historical(
    nifty_monthly_df,
    inflation_df,
    contribution_growth=0.5,
)

In [42]:
nifty_historical_full = simulate_historical(
    nifty_monthly_df,
    inflation_df,
    contribution_growth=1.0,
)

In [43]:
nifty_historical_fixed.describe()

statistic,date,price,monthly_contribution,units_bought,total_units,total_contributions,real_contributions,investment_gain,inflation_factor,nominal_corpus,real_corpus,real_wealth_created,purchasing_power_retained
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""362""",362.00,362.00,362.00,362.00,362.00,362.00,362.00,362.00,362.00,362.00,362.00,362.00
"""null_count""","""0""",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
"""mean""","""2010-12-16 05:22:12.596685""",7339.81,5000.00,1.85,500.43,907500.00,488225.72,3615997.81,3.28,4523497.81,1033505.88,545280.16,185.86
"""std""",null,6793.63,0.00,1.78,183.35,523223.18,211168.03,4203652.83,1.77,4674956.01,688190.20,502969.94,73.21
"""min""","""1995-12-01""",817.75,5000.00,0.19,5.50,5000.00,5000.00,-59954.71,1.00,5000.00,5000.00,-98204.51,69.54
"""25%""","""2003-06-01""",1480.45,5000.00,0.47,421.61,455000.00,328216.34,117650.45,1.72,478165.36,278407.80,32065.19,115.85
"""50%""","""2011-01-01""",5295.55,5000.00,0.95,576.00,910000.00,550148.76,2140470.89,2.92,3088905.99,1078634.25,526582.49,191.43
"""75%""","""2018-07-01""",10530.70,5000.00,3.38,640.01,1360000.00,669823.27,5380406.88,4.67,6705406.88,1428804.11,782759.81,230.16
"""max""","""2026-01-01""",26202.95,5000.00,6.11,668.60,1810000.00,748391.09,15709065.54,6.96,17509065.54,2633351.28,1896842.69,357.55


In [44]:
nifty_historical_half.describe()

statistic,date,price,monthly_contribution,units_bought,total_units,total_contributions,real_contributions,investment_gain,inflation_factor,nominal_corpus,real_corpus,real_wealth_created,purchasing_power_retained
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""362""",362.00,362.00,362.00,362.00,362.00,362.00,362.00,362.00,362.00,362.00,362.00,362.00
"""null_count""","""0""",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
"""mean""","""2010-12-16 05:22:12.596685""",7339.81,8829.71,2.52,636.52,1344066.55,655253.79,4617392.16,3.28,5961458.71,1338833.22,683579.43,177.97
"""std""",null,6793.63,2505.56,1.94,255.94,919723.83,325030.63,5506615.27,1.77,6370393.81,941077.45,647091.79,65.66
"""min""","""1995-12-01""",817.75,5000.00,0.50,5.50,5000.00,5000.00,-73063.56,1.00,5000.00,5000.00,-113505.51,69.79
"""25%""","""2003-06-01""",1480.45,6588.68,1.05,500.68,542360.21,386749.57,134525.37,1.72,567843.74,330622.28,36368.52,116.45
"""50%""","""2011-01-01""",5295.55,8636.20,1.60,719.06,1202799.73,705947.15,2605426.45,2.92,3844049.32,1355643.44,639343.53,183.39
"""75%""","""2018-07-01""",10530.70,10966.88,4.29,843.74,2095565.75,940382.46,6791031.80,4.67,8935706.67,1880566.58,971395.34,218.37
"""max""","""2026-01-01""",26202.95,13426.36,7.05,912.48,3196356.06,1131228.01,20712692.85,6.96,23882381.57,3577403.52,2477555.75,336.20


In [45]:
nifty_historical_full.describe()

statistic,date,price,monthly_contribution,units_bought,total_units,total_contributions,real_contributions,investment_gain,inflation_factor,nominal_corpus,real_corpus,real_wealth_created,purchasing_power_retained
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""362""",362.00,362.00,362.00,362.00,362.00,362.00,362.00,362.00,362.00,362.00,362.00,362.00
"""null_count""","""0""",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
"""mean""","""2010-12-16 05:22:12.596685""",7339.81,16415.26,3.59,824.79,2079352.13,907500.00,5994351.01,3.28,8073703.14,1774029.22,866529.22,169.75
"""std""",null,6793.63,8856.61,2.00,369.47,1686420.83,523223.18,7393113.92,1.77,9017624.90,1336305.28,851379.75,58.36
"""min""","""1995-12-01""",817.75,5000.00,1.26,5.50,5000.00,5000.00,-88492.81,1.00,5000.00,5000.00,-131010.98,69.96
"""25%""","""2003-06-01""",1480.45,8587.50,2.18,593.45,644943.92,455000.00,149092.78,1.72,673065.58,391886.83,41311.08,117.01
"""50%""","""2011-01-01""",5295.55,14602.26,2.69,899.70,1595041.87,910000.00,3170049.66,2.92,4812951.54,1697139.13,774790.80,173.61
"""75%""","""2018-07-01""",10530.70,23337.37,5.16,1137.70,3329122.77,1360000.00,8654606.62,4.67,12149301.82,2537435.51,1200306.64,209.42
"""max""","""2026-01-01""",26202.95,34793.19,9.19,1298.14,5942323.79,1810000.00,28071592.19,6.96,33945277.27,5050996.78,3320996.78,331.99


In [46]:
nifty_historical_fixed.tail(1).select(
    "date",
    "inflation_factor",
    "real_contributions",
    "real_corpus",
)

date,inflation_factor,real_contributions,real_corpus
date,f64,f64,f64
2026-01-01,6.96,748391.09,2432853.85


In [47]:
def summarize_historical(
    historical_df: pl.DataFrame,
) -> pl.DataFrame:

    return (
        historical_df
        .with_columns(
            pl.col("date").dt.year().alias("year")
        )
        .group_by("year", maintain_order=True)
        .agg(
            pl.all().last()
        )
        .drop("date")
        .select(
            "year",
            "monthly_contribution",
            "total_contributions",
            "real_contributions",
            "investment_gain",
            "nominal_corpus",
            "inflation_factor",
            "real_corpus",
            "real_wealth_created",
            "purchasing_power_retained",
        )
    )

In [48]:
historical_fixed_summary = summarize_historical(
    nifty_historical_fixed
)

historical_half_summary = summarize_historical(
    nifty_historical_half
)

historical_full_summary = summarize_historical(
    nifty_historical_full
)

In [50]:
historical_full_summary.head()

year,monthly_contribution,total_contributions,real_contributions,investment_gain,nominal_corpus,inflation_factor,real_corpus,real_wealth_created,purchasing_power_retained
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64
1995,5000.00,5000.00,5000.00,0.00,5000.00,1.00,5000.00,0.00,100.00
1996,5511.00,71132.00,65000.00,-5170.07,65961.93,1.10,59845.70,-5154.30,92.07
1997,6005.89,143202.65,125000.00,8700.39,151903.04,1.20,126461.77,1461.77,101.17
1998,6435.91,220433.57,185000.00,-23952.24,196481.33,1.29,152644.58,-32355.42,82.51
1999,7287.38,307882.13,245000.00,129660.56,437542.68,1.46,300205.75,55205.75,122.53


In [52]:
nifty_historical_fixed.write_database(
    table_name="analysis.nifty_historical_fixed",
    connection=engine,
    if_table_exists="replace",
)

-1

In [53]:
nifty_historical_half.write_database(
    table_name="analysis.nifty_historical_half",
    connection=engine,
    if_table_exists="replace",
)

-1

In [54]:
nifty_historical_full.write_database(
    table_name="analysis.nifty_historical_full",
    connection=engine,
    if_table_exists="replace",
)

-1

In [55]:
historical_fixed_summary.write_database(
    table_name="analysis.nifty_historical_fixed_summary",
    connection=engine,
    if_table_exists="replace",
)

-1

In [56]:
historical_half_summary.write_database(
    table_name="analysis.nifty_historical_half_summary",
    connection=engine,
    if_table_exists="replace",
)

-1

In [57]:
historical_full_summary.write_database(
    table_name="analysis.nifty_historical_full_summary",
    connection=engine,
    if_table_exists="replace",
)

-1

In [58]:
pl.read_database(
query = """
select table_name, column_name
from information_schema.columns
where table_schema = 'analysis'
order by table_name, ordinal_position;
""", connection = engine
        )

table_name,column_name
str,str
"""inflation""","""year"""
"""inflation""","""inflation_rate"""
"""nifty_historical_fixed""","""date"""
"""nifty_historical_fixed""","""price"""
"""nifty_historical_fixed""","""monthly_contribution"""
"""nifty_historical_fixed""","""units_bought"""
"""nifty_historical_fixed""","""total_units"""
"""nifty_historical_fixed""","""total_contributions"""
"""nifty_historical_fixed""","""real_contributions"""
